In [2]:
import numpy as np
from typing import Optional
from magssm_decoder import compute_s_ola
from transforms import get_regularised_window
import matplotlib.pyplot as plt

[path_config] AMP API : nouvelle (torch.amp.*) | sedge_src : /users/eleves-a/2023/adrien.dubois/stage/STAGE3A_UMX_MAGSSM/SEdge/src


In [3]:
L=np.arange(10)
print(L[0:-1])

[0 1 2 3 4 5 6 7 8]


In [4]:
import torch    
y = torch.randn(30)
y = y.reshape((2,3) + (5,))
print(y)
print(y.shape)

tensor([[[-1.1586,  1.9494, -0.4188,  0.1583, -0.1871],
         [-0.7953,  0.1819, -1.5942,  0.0546,  0.7377],
         [ 0.4545, -0.2564,  1.3350, -0.5300, -1.1032]],

        [[ 1.2557,  2.6433,  0.9584, -0.6017,  1.6393],
         [-1.3154,  0.1686, -0.3388, -0.2332, -0.2785],
         [ 0.4722, -0.1190, -1.4276,  0.1379,  0.6013]]])
torch.Size([2, 3, 5])


In [ ]:
def make_structured_eigenvalues(N_states, N_bins, positive_frequencies: Optional[bool] = True, domain = "frequency"):
    """ Create linearly spaced states with pre-placed imaginary parts
    """
    print("Initialisation of the eigenvalues in a structured way, positives frequency:", positive_frequencies)
    print("Domain:", domain)
    # Lambda = -0.5 + 1j * np.arange(N)

    model_rank = N_states // N_bins
    imaginary_parts = np.zeros(int(model_rank * N_bins))

    if domain == "frequency":
        targets = np.linspace(0,np.pi, N_bins)
        for i in range(N_bins):
            for j in range(model_rank):
                imaginary_parts[model_rank*i+j] = targets[i] + np.random.normal(0, np.pi/N_bins/3)
        
        remaining = np.linspace(0, np.pi, N_states - len(imaginary_parts))
        imaginary_parts = np.concatenate((imaginary_parts, remaining))
        real_parts = -1/2

    elif domain == "time":
        targets = np.arange(N_bins) #bins are time steps
        print
        for i in range(N_bins):
            for j in range(model_rank):
                imaginary_parts[model_rank*i+j] = targets[i] + np.random.normal(0, 1/3)
        
        remaining = np.linspace(0, N_bins, N_states - len(imaginary_parts))
        imaginary_parts = np.concatenate((imaginary_parts, remaining))
        real_parts = -1/2

    else:
        pass

    Lambda = real_parts + 1j * (2*int(positive_frequencies) - 1)*imaginary_parts 


    lambda_real = np.expand_dims(Lambda.real, axis=1)
    lambda_imag = np.expand_dims(Lambda.imag, axis=1)
    Lambda = np.concatenate((lambda_real, lambda_imag), axis=1)
    Lambda = torch.tensor(Lambda, dtype=torch.float)
    return Lambda


print(make_structured_eigenvalues(30,15,False, "time"))

Initialisation of the eigenvalues in a structured way, positives frequency: False
Domain: time
tensor([[ -0.5000,   0.1712],
        [ -0.5000,   0.0645],
        [ -0.5000,  -1.5314],
        [ -0.5000,  -1.0015],
        [ -0.5000,  -2.7984],
        [ -0.5000,  -1.5971],
        [ -0.5000,  -3.1263],
        [ -0.5000,  -3.1617],
        [ -0.5000,  -4.2336],
        [ -0.5000,  -3.9287],
        [ -0.5000,  -5.6044],
        [ -0.5000,  -5.4558],
        [ -0.5000,  -6.0887],
        [ -0.5000,  -5.8796],
        [ -0.5000,  -7.1949],
        [ -0.5000,  -7.0800],
        [ -0.5000,  -7.6829],
        [ -0.5000,  -8.0381],
        [ -0.5000,  -9.1427],
        [ -0.5000,  -9.0201],
        [ -0.5000,  -9.9247],
        [ -0.5000,  -9.4137],
        [ -0.5000, -10.9590],
        [ -0.5000, -10.8551],
        [ -0.5000, -12.2290],
        [ -0.5000, -11.4360],
        [ -0.5000, -12.1440],
        [ -0.5000, -12.9134],
        [ -0.5000, -13.7944],
        [ -0.5000, -14.2072]])


In [6]:
def make_linear_eigenvalues(N, symmetric = False, positive_frequencies: Optional[bool] = True):
    """ Create a S4D-Lin vector.
        Args:
            N (int32): state size
        Returns:
            N  complex eigenvalues
    """
    print('Initialisation with positive frequencies:', positive_frequencies)
    if symmetric:
        Lambda = -1/2 + (2*int(positive_frequencies) - 1)*1j * np.arange(-N//2, N//2)
    else:
        # Lambda = -0.5 + 1j * np.arange(N)
        Lambda = -1/2 + (2*int(positive_frequencies) - 1)*1j * np.linspace(0,N//2,N)

    lambda_real = np.expand_dims(Lambda.real, axis=1)
    lambda_imag = np.expand_dims(Lambda.imag, axis=1)
    Lambda = np.concatenate((lambda_real, lambda_imag), axis=1)
    Lambda = torch.tensor(Lambda, dtype=torch.float)
    return Lambda

print(make_linear_eigenvalues(10, positive_frequencies=False))

Initialisation with positive frequencies: False
tensor([[-0.5000,  0.0000],
        [-0.5000, -0.5556],
        [-0.5000, -1.1111],
        [-0.5000, -1.6667],
        [-0.5000, -2.2222],
        [-0.5000, -2.7778],
        [-0.5000, -3.3333],
        [-0.5000, -3.8889],
        [-0.5000, -4.4444],
        [-0.5000, -5.0000]])


In [ ]:
hann = torch.hann_window(40)
regularized_hann = get_regularised_window(40, epsilon1=0.5, lambda_coeff_1=4.7, lambda_coeff_2=4.7)
result = compute_s_ola(hann,300,2)
result_regularized = compute_s_ola(regularized_hann,300,2) 

result, result_regularized = result / torch.max(result), result_regularized / torch.max(result_regularized)


plt.plot(np.linspace(0,300,300),result, label="hann")
plt.plot(np.linspace(0,300,300),result_regularized, label="regularized hann")
plt.legend()
plt.savefig("/users/eleves-a/2023/adrien.dubois/stage/STAGE3A_UMX_MAGSSM/fig/tests_post_soutenance/OLA_compare_regularized.png")
plt.show()


In [13]:
from data import MUSDBDataset

dataset_kwargs = {
            "root": "/Data/adrien.dubois/musdb18_ds3",
            "is_wav": True,
            "subsets": "train",
            "target": "vocals",
            "download": False,
            "seed": 42,
        }

dataloader_kwargs = {}
valid_dataset = valid_dataset = MUSDBDataset(split="valid", samples_per_track=1, seq_duration=20, **dataset_kwargs)

valid_sampler = torch.utils.data.DataLoader(valid_dataset, batch_size=1, **dataloader_kwargs)

for x,y in valid_sampler:
    print(x, torch.max(x), torch.min(x))
    print(y, torch.max(y))
    break


torch.Size([2, 294000])
tensor([[[-0.2932, -0.2867, -0.2810,  ...,  0.0338,  0.0261,  0.0125],
         [-0.3116, -0.3020, -0.2990,  ...,  0.0671,  0.0425,  0.0721]]]) tensor(0.7087) tensor(-0.7505)
tensor([[[-0.0049, -0.0022,  0.0016,  ..., -0.0094, -0.0103, -0.0119],
         [-0.0058, -0.0030,  0.0007,  ..., -0.0016, -0.0096, -0.0205]]]) tensor(0.4134)


In [1]:
x=2
y=x
x=3
print(x,y)

3 2
